# Topic-Document Exploration
<!-- structured-notebook -->
## Notebook Summary
Purpose: inspect the documents assigned to each topic in a BERTopic model, for qualitative validation and to build intuition before running BTM.

Main steps:
- Load a saved BERTopic model and its document corpus.
- Sample representative documents per topic ordered by assignment probability.
- Identify noise / vague clusters that should be merged or flagged.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/bertopic_no_embed/` | Produced by preprocessing |
| Output | Qualitative inspection only | — |


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_MODELS,
)
# extract_text_topic.py
import os
import sys
from typing import List
import pandas as pd

# Customize these if you prefer to hardcode paths when running from an IDE:
DEFAULT_INPUT = REDDIT_MODELS / "round_11" / "data_final.csv"
DEFAULT_OUTPUT = REDDIT_MODELS / "round_11" / "text_topic.csv"

# Common aliases you might encounter
TEXT_ALIASES: List[str] = ["text", "content", "body", "post_text", "abstract"]
TOPIC_ALIASES: List[str] = ["topic", "label", "cluster", "topic_id", "theme"]

def find_column(df: pd.DataFrame, candidates: List[str]) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in cols_lower:
            return cols_lower[name]
    # Try fuzzy: strip spaces/underscores
    normalized = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}
    for name in candidates:
        key = name.replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    raise KeyError(
        f"None of the expected columns {candidates} were found. "
        f"Available columns: {list(df.columns)}"
    )

def extract_text_topic(
    input_csv: str = DEFAULT_INPUT,
    output_csv: str = DEFAULT_OUTPUT,
    encoding: str = "utf-8",
    sep: str = ",",
    dropna_text: bool = True,
) -> pd.DataFrame:
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Input CSV not found: {input_csv}")

    # Read with a couple of sensible defaults
    try:
        df = pd.read_csv(input_csv, encoding=encoding, sep=sep)
    except UnicodeDecodeError:
        # Fall back to a more permissive encoding if needed
        df = pd.read_csv(input_csv, encoding="utf-8-sig", sep=sep)

    text_col = find_column(df, TEXT_ALIASES)
    topic_col = find_column(df, TOPIC_ALIASES)

    out = df[[text_col, topic_col]].copy()
    # Normalize column names
    out.columns = ["text", "topic"]

    # Clean up whitespace
    out["text"] = out["text"].astype(str).str.strip()
    out["topic"] = out["topic"].astype(str).str.strip()

    if dropna_text:
        out = out[out["text"].str.len() > 0]

    # Save result
    out.to_csv(output_csv, index=False)
    return out


# Allow optional CLI-style args when running outside an IDE:
# python extract_text_topic.py input.csv output.csv
input_path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_INPUT
output_path = sys.argv[2] if len(sys.argv) > 2 else DEFAULT_OUTPUT

result = extract_text_topic(input_path, output_path)
print(f"Saved {len(result)} rows to '{output_path}'. Preview:")
print(result.head(10).to_string(index=False))

---
<!-- nav-footer -->

← [03 pair visuals](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/03_pair_visuals.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [05 gathering keywords](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/05_gathering_keywords.ipynb) →
